In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
RANDOM_STATE = 42

Train shape: (59, 6)
Test shape: (18, 5)

Class distribution in train.csv:
species
setosa        50
versicolor     9
Name: count, dtype: int64

NOTE: train.csv only contains classes ['setosa', 'versicolor'].
      Missing from training data: ['virginica']
      The model can only ever predict classes it has seen during training,
      so predictions will be limited to: ['setosa', 'versicolor']

Cross-validation accuracy (5-fold):
  LogisticRegression  : mean=1.0000  std=0.0000
  KNN                 : mean=0.9833  std=0.0333
  SVM (RBF)           : mean=1.0000  std=0.0000
  RandomForest        : mean=1.0000  std=0.0000

Best model: LogisticRegression (mean CV accuracy = 1.0000)

Saved predictions to submission.csv


In [ ]:
train = pd.read_csv("/content/train (1).csv")
test = pd.read_csv("/content/test (1).csv")
sample = pd.read_csv("/content/sample.csv")

features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
target = "species"

print(train.shape, test.shape)
train[target].value_counts()

In [ ]:
classes_in_train = sorted(train[target].unique())
print("classes in train:", classes_in_train)

if len(classes_in_train) < 3:
    missing = {"setosa", "versicolor", "virginica"} - set(classes_in_train)
    print("missing from train.csv:", sorted(missing))
    print("model can only predict:", classes_in_train)

X = train[features]
y = train[target]
X_test = test[features]

In [ ]:
models = {
    "log_reg": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "knn": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=5))
    ]),
    "svm": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))
    ]),
    "rf": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE))
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = {}
for name, pipe in models.items():
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")
    results[name] = scores.mean()
    print(name, "->", round(scores.mean(), 4))

best_name = max(results, key=results.get)
print("\nbest model:", best_name)

In [ ]:
best_model = models[best_name]
best_model.fit(X, y)

preds = best_model.predict(X_test)
pd.Series(preds).value_counts()

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "species": preds
})

submission.to_csv("submission.csv", index=False)
submission.head()